<a href="https://colab.research.google.com/github/Aerospace87/ML-projects/blob/main/AutoEncoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import of libraries

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [4]:
from datasets import load_dataset

In [5]:
mnist = load_dataset("mnist")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.97k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [7]:
mnist

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [11]:
mnist["train"]['image'][2]

## Transform images into tensor and shuffle

In [12]:
from torchvision import transforms

In [14]:
# The pixel values [0,255] are normalized in [0,1]
def mnist_to_tensor(samples):
  t = transforms.ToTensor()
  samples["image"] = [t(image) for image in samples["image"]]
  return samples

In [16]:
mnist = mnist.with_transform(mnist_to_tensor)
mnist["train"] = mnist["train"].shuffle(seed=1337)

## Creation of batches of tensors

In [18]:
from torch.utils.data import DataLoader

In [20]:
batch_size = 64
train_dataloader = DataLoader(mnist["train"]["image"], batch_size=batch_size)

## Convolutional Encoders

In [21]:
from torch import nn

In [23]:
def conv_block(in_channels, out_channels, kernel_size=4, stride=2, padding=1):
  """ Sequence of operations:
  1. Convolution Layer
  2. Batch Normalization Layer
  3. ReLU Activation
  """
  conv_layer = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size,
                         stride=stride, padding=padding)

  batch_norm_layer = nn.BatchNorm2d(out_channels)

  dense_layer = nn.ReLU()

  return nn.Sequential(conv_layer, batch_norm_layer, dense_layer)

In [24]:
class Encoder(nn.Module):
  def __init__(self, in_channels):
    super().__init__()
    self.conv1 = conv_block(in_channels, 128)
    self.conv2 = conv_block(128, 256)
    self.conv3 = conv_block(256, 512)
    self.conv4 = conv_block(512, 1024)
    self.linear = nn.Linear(1024, 16)

  def forward(self, x):
    """Forward propagation"""
    x = self.conv1(x) # (batch size,128, 14, 14)
    x = self.conv2(x)# (batch size, 256, 7, 7)
    x = self.conv3(x)# (batch size, 512, 3, 3)
    x = self.conv4(x)# (batch size, 1024, 1, 1)
    # Keep batch dimension when flattening
    x = self.linear(x.flatten(start_dim=1)) #(bs, 16)

    return x

In [29]:
# Because the image is in black and white only 1 channel is necessary
in_channels = 1

first_img_tensor = mnist["train"]["image"][0]

In [28]:
# The tensor size is [channels, pixels, pixels]
first_img_tensor.size()

torch.Size([1, 28, 28])

In [30]:
# Because we want to add an addition dimension due to the batchsize
x = first_img_tensor[None,:]

In [32]:
x.size()

torch.Size([1, 1, 28, 28])

In [34]:
# Eval method only for inference
encoder = Encoder(in_channels).eval()

In [35]:
encoded = encoder(x)

In [36]:
encoded.size()

torch.Size([1, 16])

In [37]:
encoded

tensor([[-0.0315, -0.0084,  0.0240,  0.0157,  0.0311,  0.0312, -0.0125, -0.0234,
          0.0027, -0.0057,  0.0294,  0.0191, -0.0027, -0.0249,  0.0109, -0.0025]],
       grad_fn=<AddmmBackward0>)